In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score, accuracy_score, mean_absolute_error

In [ ]:
import torch
import torch.nn as nn 
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim 
from torch.optim.lr_scheduler import ReduceLROnPlateau


In [ ]:
import random
import copy
from collections import defaultdict


In [ ]:
df_app = pd.read_csv('App_FeatureSelection.csv')

In [ ]:
df_app = df_app.sort_values(by = ['user_id', 'date'])
df_app['date'] = pd.to_datetime(df_app['date'])
df_app['day_order'] = (df_app.date - df_app.groupby('user_order')['date'].transform('min')).dt.days + 1
df_app['date_diff'] = df_app.groupby('user_order')['date'].diff().dt.days

df_app = df_app.drop(index = [1124, 1125, 1126])
df_app = df_app[df_app['user_order'] != 35]
df_app = df_app.reset_index(drop = True)

In [ ]:
df_app = df_app.groupby('user_order').head(27).reset_index(drop = True)

In [ ]:
df_app.groupby('user_order').size().values

In [ ]:
df_app

In [ ]:
feature_list = ['phq1', 'phq2', 'meal_morning', 'meal_lunch', 'meal_dinner',
       'meal_night', 'sleep_satisfy', 'sleep_disturb', 'amount_beer_glass',
       'amount_soju_glass', 'sleep_minute_adjusted']

target_variable = 'tmrw_alcohol_desire'

In [ ]:
X = df_app[feature_list]
y = df_app[target_variable]

In [ ]:
X_train, y_train, user_train = [], [], []
X_val, y_val, user_val = [], [], []
X_test, y_test, user_test = [], [], []

In [ ]:
window = 4
val_days = 3
test_days = 5

In [ ]:
for user, user_data in df_app.groupby('user_order'):
    
    
    X_user = user_data[feature_list].to_numpy(dtype = np.float32)
    y_user = user_data[target_variable].to_numpy()
    
    T = len(user_data)
    
    # Index 기준: Starts from 0
    test_start_day = T - test_days
    val_start_day = T - (val_days + test_days)
    train_end_day = val_start_day - 1 
    
    
    
    for start in range(0, T - window + 1):
        
        target_idx = start + window - 1
        
        X_window = X_user[start : start + window]
        y_window = y_user[target_idx] 
        y_window = int(y_window) - 1 # [1,2,3,4,5] -> [0,1,2,3,4]
        
        
        # 1. Check Within Window
        day_list = user_data['day_order'].iloc[start : start + window]
        day_array = np.array(day_list)
        flag = (np.diff(day_array) == 1).all()
        
        if flag != True:
            continue
        # ==================================================================
        
        # 2. Check Next Day Target
        if target_idx + 1 < T:
            last_input_day = user_data['day_order'].iloc[target_idx]
            next_day = user_data['day_order'].iloc[target_idx + 1]
        
            if next_day - last_input_day != 1:
                continue
        # ==================================================================
        
        # Train, Val, Test Split
        if target_idx <= train_end_day:
            X_train.append(X_window)
            y_train.append(y_window)
            user_train.append(user)

            
        elif val_start_day <= target_idx < test_start_day:
            X_val.append(X_window)
            y_val.append(y_window)
            user_val.append(user)

            
        elif test_start_day <= target_idx:
            X_test.append(X_window)
            y_test.append(y_window)
            user_test.append(user)

        
        
        

### list -> Numpy()

In [ ]:
X_train, y_train, user_train = np.array(X_train), np.array(y_train), np.array(user_train)
X_val, y_val, user_val = np.array(X_val), np.array(y_val), np.array(user_val)
X_test, y_test, user_test = np.array(X_test), np.array(y_test), np.array(user_test)

### numpy() -> pytorch Tensor

In [ ]:
X_train_tensor = torch.from_numpy(X_train)
y_train_tensor = torch.from_numpy(y_train).long()

X_val_tensor = torch.from_numpy(X_val)
y_val_tensor = torch.from_numpy(y_val).long()

X_test_tensor = torch.from_numpy(X_test)
y_test_tensor = torch.from_numpy(y_test).long()

user_train_tensor = torch.from_numpy(user_train)
user_val_tensor = torch.from_numpy(user_val)
user_test_tensor = torch.from_numpy(user_test)


In [ ]:
X_train_tensor.shape

In [ ]:
951 / 32

y_train_tensor.shape

In [ ]:
X_val_tensor.shape

In [ ]:
X_test_tensor.shape

In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor, user_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor, user_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor, user_test_tensor)

## DataLoader

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle= True, drop_last= False)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size = 256, shuffle=False, drop_last=False)

### Vanilla RNN Model

In [ ]:
class TinyVanillaRNN(nn.Module):
    
    def __init__(self, input_size, hidden_size, task = 'cls', num_classes = 5):
        super().__init__()
        self.task = task 
        self.rnn = nn.RNN(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers = 1,
            batch_first = True,
            nonlinearity = 'tanh'
        )
        
        self.head = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        # x : (Batch, Window, Feature)
        out, h_n = self.rnn(x)
        h_last = h_n[-1]
        y_hat = self.head(h_last)
        
        return y_hat 

In [ ]:
window_size_list = [2,4,6,8]

In [ ]:
hidden_size_list = [8,16,32]

## Run 100 Times

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [ ]:
input_size = X_train_tensor.shape[2]
hidden_size = hidden_size_list[1]
EPOCHS = 70

In [ ]:
@torch.no_grad()
def eval_metrics_on_loader(model, loader, criterion):
    
    model.eval() # Evaluation Mode
    
    # Intialize Dummy Values
    total = 0 
    correct = 0
    loss_sum = 0.0 
    
    # Initialize Lists for Summary
    all_pred = [] # Predicted Values
    all_true = [] # True Values
    all_user = [] # User Values
    all_day = [] # Day Information
    
    
    for X_batch, y_batch, user_batch in loader:
        
        logits = model(X_batch)
        pred = logits.argmax(dim = 1)
        
        loss = criterion(logits, y_batch)
        loss_sum = loss_sum + loss.item() * X_batch.size(0) # Get Loss Sum
        
        correct = correct + (pred == y_batch).sum().item()
        total = total + y_batch.size(0)
        
        # Pytorch Tensor -> Numpy() -> STAN Code의 Data Part와 대응됨
        all_pred.append(pred.detach().numpy())
        all_true.append(y_batch.detach().numpy())
        all_user.append(user_batch.detach().numpy())
        
        
        
    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)
    users = np.concatenate(all_user)
    # days = np.concatenate(all_day)
    
    # Participant Wise Accuracy
    user_correct = defaultdict(int)
    user_total = defaultdict(int)
    
    for u, y_p, y_t in zip(users, y_pred, y_true):
        # Update User Total
        user_total[u] += 1 
        
        # Update User Correct if actually correct
        if y_p == y_t:
            user_correct[u] += 1
    
    # Calculate User Accuracy
    user_acc = {u: user_correct[u] / user_total[u] for u in user_total}
    
    
    metrics = {
        'loss' : (loss_sum / total),
        'acc' : correct / total,
        
        
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "mae": mean_absolute_error(y_true, y_pred),
        
        'user_acc' : user_acc,
    }
    
    return metrics

In [ ]:
def train_one_run(run_id, seed):
    
    # Set Seed
    set_seed(seed)

    # Define a New Model each Time
    model = TinyVanillaRNN(input_size=input_size, hidden_size=hidden_size, task='cls', num_classes=5)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    history = []

    # 2) train loop
    for epoch in range(EPOCHS):
        model.train()
        
        total_loss = 0.0
        train_correct = 0
        train_total = 0
        
        
        for X_batch, y_batch, user_batch in train_loader:


            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            
            total_loss = total_loss + loss.item() * X_batch.size(0)
            pred = logits.argmax(dim = 1)
            train_correct += (pred == y_batch).sum().item()
            train_total += y_batch.size(0)
            
        
        
        train_metrics = eval_metrics_on_loader(model, train_loader, criterion=criterion)
        val_metrics   = eval_metrics_on_loader(model, val_loader,   criterion=criterion)

    # 3) test preds/labels
        history.append({
                "run": run_id,
                "seed": seed,
                "epoch": epoch + 1,
                "train_loss": train_metrics["loss"],
                "train_acc": train_metrics["acc"],
                "train_macro_f1": train_metrics["macro_f1"],
                "train_weighted_f1": train_metrics["weighted_f1"],
                "train_mae": train_metrics["mae"],
                "val_loss": val_metrics["loss"],
                "val_acc": val_metrics["acc"],
                "val_macro_f1": val_metrics["macro_f1"],
                "val_weighted_f1": val_metrics["weighted_f1"],
                "val_mae": val_metrics["mae"],
            })
        
    # Final Evaluation
    train_metrics = eval_metrics_on_loader(model, train_loader, criterion=criterion)
    val_metrics   = eval_metrics_on_loader(model, val_loader,   criterion=criterion)
    test_metrics = eval_metrics_on_loader(model, test_loader, criterion=criterion)
        
    summary = {
        "run": run_id,
        "seed": seed,

        # Train (final model)
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "train_macro_f1": train_metrics["macro_f1"],
        "train_weighted_f1": train_metrics["weighted_f1"],
        "train_macro_precision": train_metrics["macro_precision"],
        "train_macro_recall": train_metrics["macro_recall"],
        "train_mae": train_metrics["mae"],

        # Val (final model)
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_weighted_f1": val_metrics["weighted_f1"],
        "val_macro_precision": val_metrics["macro_precision"],
        "val_macro_recall": val_metrics["macro_recall"],
        "val_mae": val_metrics["mae"],

        # Test (final model)
        "test_loss": test_metrics["loss"],
        "test_acc": test_metrics["acc"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_weighted_f1": test_metrics["weighted_f1"],
        "test_macro_precision": test_metrics["macro_precision"],
        "test_macro_recall": test_metrics["macro_recall"],
        "test_mae": test_metrics["mae"],
    }



    return summary, history


In [ ]:
def train_one_run(run_id, seed):
    
    # Set Seed
    set_seed(seed)
    
    # Define a Model
    model = TinyVanillaRNN(input_size= input_size, hidden_size=hidden_size, num_classes=5)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
    
    history = []
    
    # Define Settings for finding the best Epoch for val-loss
    best_val_loss = float('inf') # Set Initial Value to Infinity
    best_epoch = None
    best_state = None
    
    for epoch in range(EPOCHS):
        
        model.train() # Training Mode
        
        total_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for X_batch, y_batch, user_batch in train_loader:
            
            optimizer.zero_grad() # Initialize Weights
            logits = model(X_batch)
            loss = criterion(logits, y_batch) # Calculate Loss
            loss.backward() # Back Propagation -> Calculate Derivatives
            optimizer.step() # Update Weights
            
            total_loss = total_loss + loss.item() * X_batch.size(0)
            pred = logits.argmax(dim = 1)
            train_correct = train_correct + (pred == y_batch).sum().item()
            train_total = train_total + y_batch.size(0)
            
        train_metrics = eval_metrics_on_loader(model, train_loader, criterion=criterion)
        val_metrics = eval_metrics_on_loader(model, val_loader, criterion=criterion)
        
    
        # Append to the History List
        history.append({
            "run": run_id,
            "seed": seed,
            
            "epoch": epoch + 1,
            
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "train_mae": train_metrics["mae"],
            
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "val_mae": val_metrics["mae"],
        })    
        
        # Update Best Checkpoint
        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())
        
    # Load Weights of the best checkpoint
    model.load_state_dict(best_state)    
    
    # After Total Model Training => Final Evaluation
    final_train_metrics = eval_metrics_on_loader(model, train_loader, criterion = criterion)
    final_val_metrics   = eval_metrics_on_loader(model, val_loader,   criterion=criterion)
    final_test_metrics = eval_metrics_on_loader(model, test_loader, criterion=criterion)


    # Create A Summary
    summary = {
        "run": run_id,
        "seed": seed,
        
        # Add Best Epoch, Best Val_loss
        'best_epoch' : best_epoch,
        'best_val_loss': best_val_loss,
        
        
        # Train (final model)
        "train_loss": final_train_metrics["loss"],
        "train_acc": final_train_metrics["acc"],
        "train_macro_f1": final_train_metrics["macro_f1"],
        "train_weighted_f1": final_train_metrics["weighted_f1"],
        "train_macro_precision": final_train_metrics["macro_precision"],
        "train_macro_recall": final_train_metrics["macro_recall"],
        "train_mae": final_train_metrics["mae"],

        # Val (final model)
        "val_loss": final_val_metrics["loss"],
        "val_acc": final_val_metrics["acc"],
        "val_macro_f1": final_val_metrics["macro_f1"],
        "val_weighted_f1": final_val_metrics["weighted_f1"],
        "val_macro_precision": final_val_metrics["macro_precision"],
        "val_macro_recall": final_val_metrics["macro_recall"],
        "val_mae": final_val_metrics["mae"],

        # Test (final model)
        "test_loss": final_test_metrics["loss"],
        "test_acc": final_test_metrics["acc"],
        "test_macro_f1": final_test_metrics["macro_f1"],
        "test_weighted_f1": final_test_metrics["weighted_f1"],
        "test_macro_precision": final_test_metrics["macro_precision"],
        "test_macro_recall": final_test_metrics["macro_recall"],
        "test_mae": final_test_metrics["mae"],
    
    }
    
    # # Participant Wise Accuracy
    # train_user_acc = final_train_metrics['user_acc']
    # val_user_acc = final_val_metrics['user_acc']
    # test_user_acc = final_test_metrics['user_acc']
    
    # # Get Predicted Alcohol Desire Values for each day
    # df_train_pred = predict_on_loader(model, train_loader)
    # df_val_pred = predict_on_loader(model, val_loader)
    # df_test_pred = predict_on_loader(model, test_loader)
    
    return summary, history
      

In [ ]:
N_RUNS = 100
base_seed = 2026
seeds = [base_seed + i for i in range(N_RUNS)]

In [ ]:
all_summaries = []
all_histories = []

In [ ]:
for id, seed in enumerate(seeds):
    summary, history = train_one_run(run_id=id, seed=seed)
    all_summaries.append(summary)
    all_histories.extend(history)

df_runs = pd.DataFrame(all_summaries)    
df_hist = pd.DataFrame(all_histories) 

In [ ]:
df_runs

In [ ]:
df_runs.columns

In [ ]:
cols = ["train_acc", "train_macro_f1", "train_weighted_f1", "train_mae",
        "val_acc", "val_macro_f1", "val_weighted_f1", "val_mae",
        "test_acc", "test_macro_f1", "test_weighted_f1", "test_mae",]

In [ ]:
df_runs[cols].agg(["mean","std"]).T.round(4)